In [1]:
import os
print("Current working directory:", os.getcwd())

Current working directory: /Users/nullset/Developer/ML/learning-project/notebooks


In [2]:
import sys
import os
repo_root = "/Users/nullset/Developer/ML/learning-project"

print("Repo root:", repo_root)
if repo_root not in sys.path:
    sys.path.append(repo_root)

from src import features  # noqa: E402


Repo root: /Users/nullset/Developer/ML/learning-project


In [3]:
raw_path = os.path.join(repo_root, "data", "raw")
interim_path = os.path.join(repo_root, "data", "interim")

features.process_folder(raw_path, interim_path, augment=True)

In [4]:
import os
import shutil
import random

processed_path = os.path.join(repo_root, "data", "processed")

def split_dataset(interim_dir, processed_dir, train_ratio=0.7, val_ratio=0.15):
    os.makedirs(processed_dir, exist_ok=True)
    for split in ["train", "val", "test"]:
        os.makedirs(os.path.join(processed_dir, split), exist_ok=True)

    for class_name in os.listdir(interim_dir):
        class_path = os.path.join(interim_dir, class_name)
        if not os.path.isdir(class_path):
            continue

        files = [
            f for f in os.listdir(class_path)
            if os.path.isfile(os.path.join(class_path, f)) and f.lower().endswith(".npy")
        ]
        random.shuffle(files)

        n = len(files)
        if n == 0:
            continue

        n_train = int(train_ratio * n)
        n_val = int(val_ratio * n)

        splits = {
            "train": files[:n_train],
            "val": files[n_train:n_train+n_val],
            "test": files[n_train+n_val:]
        }

        for split, split_files in splits.items():
            split_class_dir = os.path.join(processed_dir, split, class_name)
            os.makedirs(split_class_dir, exist_ok=True)
            for f in split_files:
                src = os.path.join(class_path, f)
                dst = os.path.join(split_class_dir, f)
                shutil.copy(src, dst)

split_dataset(interim_path, processed_path)
print("Dataset split complete")

Dataset split complete


In [5]:
import numpy as np

def load_npy_dataset(folder):
    X = []
    y = []

    for class_name in os.listdir(folder):
        class_path = os.path.join(folder, class_name)
        if not os.path.isdir(class_path):
            continue

        for file in os.listdir(class_path):
            if file.endswith(".npy"):
                X.append(np.load(os.path.join(class_path, file)))
                y.append(int(class_name))

    return np.array(X), np.array(y)

X_train, y_train = load_npy_dataset(os.path.join(processed_path, "train"))
X_val, y_val = load_npy_dataset(os.path.join(processed_path, "val"))

In [6]:
from tensorflow.keras import layers, models # type: ignore

model = models.Sequential([
    layers.Input(shape=(64,64,3)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(3, activation='softmax')
])

In [7]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=10)

Epoch 1/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 125ms/step - accuracy: 0.2459 - loss: 1.3426 - val_accuracy: 0.2857 - val_loss: 1.1399
Epoch 2/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.3934 - loss: 1.0891 - val_accuracy: 0.3214 - val_loss: 1.0854
Epoch 3/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.3443 - loss: 1.0750 - val_accuracy: 0.4643 - val_loss: 1.0685
Epoch 4/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.4754 - loss: 1.0428 - val_accuracy: 0.2857 - val_loss: 1.0426
Epoch 5/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - accuracy: 0.4098 - loss: 0.9829 - val_accuracy: 0.4643 - val_loss: 1.0089
Epoch 6/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.5246 - loss: 0.9285 - val_accuracy: 0.5714 - val_loss: 0.9292
Epoch 7/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.7541 - loss: 0.8478 - val_accuracy: 0.7500 - val_loss: 0.8599
Epoch 8/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.8197 - loss: 0.7668 - val_accuracy: 0.7500 - val_loss: 0.8016

In [8]:
model.save(os.path.join(repo_root, "models", "hand_sign_cnn.h5"))

In [9]:
X_test, y_test = load_npy_dataset(os.path.join(processed_path, "test"))

model.evaluate(X_test, y_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8667 - loss: 0.5816


[0.5815966725349426, 0.8666666746139526]

In [10]:
sample = X_test[5]
pred = model.predict(sample.reshape(1,64,64,3))

print("Predicted:", np.argmax(pred))
print("Actual:", y_test[5])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
Predicted: 0
Actual: 0


In [11]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       802,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,467,403 (9.41 MB)

 Trainable params: 822,467 (3.14 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,644,936 (6.27 MB)